# PersonaPlex + IMTalker Live Avatar with Emotion - RunPod

Runs the **emotion-enabled** live avatar server. This is the stock AH
AudioPace winner server (PersonaPlex + IMTalker) started through a thin
wrapper (`emotion_finetune/launch_emotion_live.py`) that applies your trained
emotion adapter to the FM generator.

Set `AVATAR_EMOTION` to control the avatar's expression for the whole
conversation:

| `AVATAR_EMOTION` | behavior |
|---|---|
| `neutral` (default) | original IMTalker behavior (bit-identical) |
| `happy` | consistent happy expression |
| `sad` | consistent sad expression |
| `angry` | consistent angry expression |
| `surprise` | consistent surprised expression |
| `fear` | consistent fearful expression |

The emotion enters the IMTalker conditioning as a fifth additive term; the
PersonaPlex reply generation, the RMS gate, the timing contract, and the
renderer are all unchanged. With no adapter configured the server is
bit-identical to the stock AH server.

**Prerequisite:** a trained `emotion_adapter.ckpt` from
`RunPod_Emotion_Training.ipynb` (or leave `EMOTION_ADAPTER_PATH` empty to run
the stock neutral server).


## Parameters

Edit these before running.

In [ ]:
import os

# --- Project location -------------------------------------------------------
PROJECT_ROOT = "/workspace/speech2avatar"
GIT_REPO_URL = "https://github.com/nash-raf/speech2avatar.git"  # leave blank if already cloned
GIT_BRANCH = "main"
VENV_DIR = "/workspace/preprocess_5090"

# --- Service ------------------------------------------------------------------
HOST = "0.0.0.0"
PORT = 8998

RUNPOD_POD_ID = os.environ.get("RUNPOD_POD_ID", "")
PUBLIC_URL = f"https://{RUNPOD_POD_ID}-{PORT}.proxy.runpod.net/" if RUNPOD_POD_ID else None

# --- Emotion --------------------------------------------------------------------
# Local path to a trained emotion adapter, OR an HF spec "repo_id:filename"
# (downloaded in Step 9). Leave EMPTY to run the stock neutral server.
EMOTION_ADAPTER_PATH = ""          # e.g. f"{PROJECT_ROOT}/checkpoints/emotion_adapter_final.ckpt"
EMOTION_ADAPTER_HF = ""            # e.g. "your-hf-user/imtalker-emotion:emotion_adapter_final.ckpt"
AVATAR_EMOTION = "neutral"         # neutral | happy | sad | angry | surprise | fear
EMOTION_SCALE = "1.0"              # emotion intensity multiplier; if the expression is
                                   # too subtle try "2" or "4" (no retrain needed)
EMOTION_LORA_RANK = "16"           # must match the adapter's training rank
EMOTION_LORA_ALPHA = "32"          # must match the adapter's training alpha

# --- Hugging Face sources (must match live.md) -----------------------------------
HF_ADAPTER_REPO = "niloy629/hdtf_preprocess"
HF_ADAPTER_FILE = "personaplex_lookahead_rms_adapter/checkpoints/personaplex_lookahead096_future048_rms50_adapter.pt"
HF_SILENCE_FILE = "personaplex_lookahead_rms_adapter/stats/silence_helium_mean.pt"
HF_LORA_REPO = "niloy629/hdtf_preprocess"
HF_LORA_FILE = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
HF_LORA_SHA256 = "f34de1bd748857bc802102da578046bb10bd5f664460607d9c785ff35922e52f"
HF_IMTALKER_REPO = "cbsjtu01/IMTalker"
HF_PERSONAPLEX_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"
HF_PERSONAPLEX_MIMI_REPO = "nvidia/personaplex-7b-v1"

EXPECTED_TORCH_VERSION = "2.8.0+cu128"

# --- Optional run_live_emotion.sh overrides (leave "" for script defaults) --------
REF_PATH = ""
A_CFG_SCALE = ""
NFE = ""
VOICE_PROMPT = ""
DISABLE_LORA = ""
ENABLE_SEARCH = False
WEB_SEARCH_ENABLED = False
WEB_SEARCH_API_KEY = os.environ.get("WEB_SEARCH_API_KEY", "")
ROUTER_THRESHOLD = "0.40"
ROUTER_RULES = "1"

STARTUP_TIMEOUT_SEC = 900
POLL_INTERVAL_SEC = 5

# --- Derived paths ------------------------------------------------------------------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
PERSONAPLEX_DIR = f"{PROJECT_ROOT}/personaplex"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
EMOTION_DIR = f"{PROJECT_ROOT}/emotion_finetune"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"
LOG_PATH = f"{PROJECT_ROOT}/live_server_emotion.log"
PID_PATH = f"{PROJECT_ROOT}/.run_live_emotion.pid"

os.makedirs("/workspace", exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"HOST:PORT    = {HOST}:{PORT}")
print(f"AVATAR_EMOTION = {AVATAR_EMOTION}  (scale={EMOTION_SCALE})")
print(f"EMOTION_ADAPTER_PATH = {EMOTION_ADAPTER_PATH or EMOTION_ADAPTER_HF or '(none - stock neutral server)'}")
if PUBLIC_URL:
    print(f"PUBLIC_URL = {PUBLIC_URL}")


## Utilities

In [ ]:
import hashlib
import json as _json
import os
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),"
        "'device_count': torch.cuda.device_count()"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def assert_torch_pin(auto_fix=True):
    info = get_torch_info()
    print(f"[info] torch={info['version']} cuda={info['cuda_available']} "
          f"cuda_version={info['cuda_version']} device={info['device_name']} "
          f"count={info['device_count']}")
    if info["version"] != EXPECTED_TORCH_VERSION:
        print(f"[warn] torch drifted to '{info['version']}' (expected '{EXPECTED_TORCH_VERSION}')")
        if not auto_fix:
            raise RuntimeError(f"torch pin violated: {info['version']}")
        print("[recovery] reinstalling pinned torch (cu128, no-deps)...")
        run(
            f"{VENV_ACTIVATE} && pip install --force-reinstall --no-deps "
            f"torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
            f"--index-url https://download.pytorch.org/whl/cu128",
            retries=3,
        )
        info = get_torch_info()
        if info["version"] != EXPECTED_TORCH_VERSION:
            raise RuntimeError(f"torch pin recovery failed; still got {info['version']}")
    print(f"[ok] torch pin confirmed: {info['version']}")
    return info


def sha256_of(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def hf_download(repo, filename, local_dir, repo_type="model", retries=3):
    repo_type_flag = f"--repo-type {repo_type}" if repo_type != "model" else ""
    os.makedirs(local_dir, exist_ok=True)
    cmd = (
        f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
        f"hf download {repo} {filename} {repo_type_flag} --local-dir {local_dir}"
    )
    run(cmd, retries=retries, retry_delay=15)


def tail_file(path, n=40):
    if not os.path.exists(path):
        print(f"(no {path} yet)")
        return
    with open(path, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


## Step 0 - Clone / validate the project

In [ ]:
# Step 0 - clone (if needed) and validate the project structure.
if GIT_REPO_URL and not os.path.isdir(os.path.join(PROJECT_ROOT, "IMTalker")):
    run(f"git clone {GIT_REPO_URL} {PROJECT_ROOT}")
    if GIT_BRANCH:
        run(f"git checkout {GIT_BRANCH}", cwd=PROJECT_ROOT)

REQUIRED_STRUCTURE = [
    "IMTalker",
    "IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py",
    "IMTalker/liveTry.py",
    "IMTalker/ws_av_binary_codec.py",
    "IMTalker/static/index_v3_binary_fullscreen_aj_nodrop.html",
    "scripts/download_live_assets.sh",
    "emotion_finetune",
    "emotion_finetune/emotion_modules.py",
    "emotion_finetune/launch_emotion_live.py",
    "run_live_emotion.sh",
]
missing = [p for p in REQUIRED_STRUCTURE if not os.path.exists(os.path.join(PROJECT_ROOT, p))]
if missing:
    raise RuntimeError("project structure incomplete; missing:\n  " + "\n  ".join(missing))
run("chmod +x run_live_emotion.sh scripts/download_live_assets.sh", cwd=PROJECT_ROOT, check=False)
print("[ok] project structure validated")


## Step 1 - System dependencies

In [ ]:
# Step 1 - system dependencies (matches live.md Fresh Pod Setup).
run(
    "export DEBIAN_FRONTEND=noninteractive && apt-get update && "
    "apt-get install -y python3.11 python3.11-venv ffmpeg git",
    retries=2,
)


## Step 2 - GPU / driver check

In [ ]:
# Step 2 - GPU / driver check.
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash",
                     capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    raise RuntimeError("nvidia-smi failed - this notebook needs a CUDA GPU pod")


## Step 3 - Python 3.11 venv

In [ ]:
# Step 3 - Python 3.11 venv.
run(f"python3.11 -m venv {VENV_DIR}")
run(f"{VENV_ACTIVATE} && python -m pip install --upgrade pip wheel")
run(f"{VENV_ACTIVATE} && python -m pip install setuptools==80.9.0")


## Step 4 - Pinned PyTorch (cu128)

In [ ]:
# Step 4 - pinned PyTorch (cu128, required for RTX 5090 / sm_120).
run(
    f"{VENV_ACTIVATE} && pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
    f"--index-url https://download.pytorch.org/whl/cu128",
    retries=3,
)
torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError("CUDA is not available inside the venv")


## Step 5 - IMTalker Python dependencies

In [ ]:
# Step 5 - IMTalker Python dependencies.
run(f"{VENV_ACTIVATE} && pip install -r {IMTALKER_DIR}/requirement.txt", retries=2)


## Step 6 - Hugging Face tooling + streaming dependencies

In [ ]:
# Step 6 - Hugging Face tooling + streaming/codec dependencies.
run(f'{VENV_ACTIVATE} && pip install "huggingface_hub[cli]" hf_transfer tensorboard', retries=2)
run(
    f'{VENV_ACTIVATE} && pip install "sphn>=0.2.0,<0.3.0" einops sentencepiece aiohttp av aiortc bitsandbytes',
    retries=2,
)


## Step 7 - Hugging Face authentication

In [ ]:
# Step 7 - Hugging Face authentication.
import getpass

token = os.environ.get("HF_TOKEN", "").strip()
if not token:
    token = getpass.getpass("Hugging Face token (read access): ").strip()
os.environ["HF_TOKEN"] = token
run(f"{VENV_ACTIVATE} && hf auth login --token $HF_TOKEN", quiet=True)
print("[ok] Hugging Face authenticated")


## Step 8 - Download runtime assets

In [ ]:
# Step 8 - download the runtime assets (the project's own script).
run(
    f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
    f"export SPEECH2AVATAR_ROOT={PROJECT_ROOT} && "
    f"bash scripts/download_live_assets.sh",
    cwd=PROJECT_ROOT,
    retries=2,
)


## Step 9 - Install PersonaPlex / Moshi

In [ ]:
# Step 9 - install the PersonaPlex/Moshi package (--no-deps, per live.md).
moshi_candidates = [
    os.path.join(PERSONAPLEX_BNB4_DIR, "moshi"),
    os.path.join(PERSONAPLEX_DIR, "moshi"),
]
moshi_dir = next((p for p in moshi_candidates if os.path.isdir(p)), None)
if moshi_dir is None:
    raise RuntimeError(f"moshi package not found in {moshi_candidates}")
run(f"{VENV_ACTIVATE} && pip install -e {moshi_dir} --no-deps", retries=2)
print(f"[ok] moshi installed from {moshi_dir}")


## Step 10 - Resolve the emotion adapter

In [ ]:
# Step 10 - resolve the emotion adapter (local path or Hugging Face).
if EMOTION_ADAPTER_HF and not EMOTION_ADAPTER_PATH:
    repo_id, _, fname = EMOTION_ADAPTER_HF.partition(":")
    if not repo_id or not fname:
        raise RuntimeError("EMOTION_ADAPTER_HF must be 'repo_id:filename'")
    EMOTION_ADAPTER_PATH = f"{CHECKPOINT_DIR}/emotion/{fname}"
    if not os.path.exists(EMOTION_ADAPTER_PATH):
        hf_download(repo_id, fname, f"{CHECKPOINT_DIR}/emotion", repo_type="model")

if EMOTION_ADAPTER_PATH:
    if not os.path.exists(EMOTION_ADAPTER_PATH):
        raise RuntimeError(f"emotion adapter not found: {EMOTION_ADAPTER_PATH}")
    size_mb = os.path.getsize(EMOTION_ADAPTER_PATH) / 1e6
    print(f"[ok] emotion adapter: {EMOTION_ADAPTER_PATH} ({size_mb:.1f} MB)")
    print(f"     avatar emotion: {AVATAR_EMOTION} (scale={EMOTION_SCALE})")
else:
    if AVATAR_EMOTION.lower() not in ("neutral", "none", ""):
        print(f"[warn] AVATAR_EMOTION={AVATAR_EMOTION} but no adapter configured - "
              "the server will run neutral")
    print("[ok] no emotion adapter - stock neutral AH server")


## Step 11 - Launch the emotion-enabled live server

Launches `run_live_emotion.sh` in the background (it sources the venv, resolves
the checkpoints, exports the emotion configuration, and starts the stock AH
server through the emotion wrapper). The emotion is read from the environment:

* `EMOTION_ADAPTER_PATH` - trained adapter (empty = stock neutral server)
* `AVATAR_EMOTION` - constant expression for the whole conversation
* `EMOTION_SCALE` - intensity multiplier


In [ ]:
env_overrides = {
    "HOST": HOST,
    "PORT": str(PORT),
    "VENV_DIR": VENV_DIR,
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
    "PROJECT_ROOT": PROJECT_ROOT,
    "EMOTION_ADAPTER_PATH": EMOTION_ADAPTER_PATH,
    "AVATAR_EMOTION": AVATAR_EMOTION,
    "EMOTION_SCALE": str(EMOTION_SCALE),
    "EMOTION_LORA_RANK": str(EMOTION_LORA_RANK),
    "EMOTION_LORA_ALPHA": str(EMOTION_LORA_ALPHA),
}
if REF_PATH:
    env_overrides["REF_PATH"] = REF_PATH
if A_CFG_SCALE:
    env_overrides["A_CFG_SCALE"] = str(A_CFG_SCALE)
if NFE:
    env_overrides["NFE"] = str(NFE)
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if DISABLE_LORA:
    env_overrides["DISABLE_LORA"] = str(DISABLE_LORA)
if ENABLE_SEARCH:
    env_overrides["ENABLE_SEARCH"] = "1"
    env_overrides["ROUTER_THRESHOLD"] = str(ROUTER_THRESHOLD)
    env_overrides["ROUTER_RULES"] = str(ROUTER_RULES)
    if WEB_SEARCH_ENABLED:
        env_overrides["WEB_SEARCH_ENABLED"] = "1"
        env_overrides["WEB_SEARCH_API_KEY"] = WEB_SEARCH_API_KEY
    else:
        env_overrides["WEB_SEARCH_ENABLED"] = "0"

print("Resolved environment overrides for run_live_emotion.sh:")
for k, v in env_overrides.items():
    print(f"  {k}={'***' if k == 'WEB_SEARCH_API_KEY' else v}")

import subprocess
launch_env = os.environ.copy()
launch_env.update(env_overrides)
log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", "run_live_emotion.sh"], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))
print(f"[ok] launched emotion live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")


## Step 12 - Wait for healthy startup

In [ ]:
# Step 12 - wait for healthy startup.
import time

HEALTHY_MARKERS = {
    "lora loaded": "lora loaded",
    "using direct Moshi reply hidden": "using direct Moshi reply hidden",
    "installed PersonaPlex graphed hidden capture": "installed PersonaPlex graphed hidden capture",
    "serving": "serving",
}
EMOTION_MARKERS = ("[emotion] live emotion enabled", "[emotion] disabled")

start = time.time()
seen = set()
emotion_seen = False
while time.time() - start < STARTUP_TIMEOUT_SEC:
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log = f.read()
        for key, marker in HEALTHY_MARKERS.items():
            if marker in log and key not in seen:
                seen.add(key)
                print(f"[ok] {key}")
        if not emotion_seen and any(m in log for m in EMOTION_MARKERS):
            emotion_seen = True
            for line in log.splitlines():
                if "[emotion]" in line:
                    print(f"[ok] {line.strip()}")
        if "Uvicorn running on" in log:
            print("[ok] Uvicorn running")
            break
        if "Traceback" in log or "Missing required path" in log:
            print("[error] startup failed - last log lines:")
            tail_file(LOG_PATH, 40)
            raise RuntimeError("live server startup failed (see log above)")
    time.sleep(POLL_INTERVAL_SEC)
else:
    print("[error] startup timed out - last log lines:")
    tail_file(LOG_PATH, 40)
    raise RuntimeError(f"live server did not become healthy within {STARTUP_TIMEOUT_SEC}s")

missing = set(HEALTHY_MARKERS) - seen
if missing:
    print(f"[warn] markers not seen yet: {missing}")
print("[ok] live server is healthy")


## Step 13 - HTTP / Uvicorn confirmation

In [ ]:
# Step 13 - HTTP / Uvicorn confirmation.
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10) as resp:
        print(f"[ok] HTTP {resp.status} from http://127.0.0.1:{PORT}/")
except Exception as exc:
    print(f"[warn] HTTP check failed: {exc}")

if PUBLIC_URL:
    print(f"\\nOpen the avatar UI: {PUBLIC_URL}")
    print(f"Avatar emotion: {AVATAR_EMOTION}")


## Operational cells (run any time)

To **change the emotion**, stop the server (cell below), set a new
`AVATAR_EMOTION` in the parameters cell, and re-run the launch + health cells.


In [ ]:
# Tail the live server log.
tail_file(LOG_PATH, 200)


In [ ]:
# Stop switch - set STOP_SERVER = True and re-run to terminate the live server.
STOP_SERVER = False

if not STOP_SERVER:
    print("set STOP_SERVER = True and re-run this cell to stop the live server.")
else:
    import os, signal
    with open(PID_PATH) as f:
        pid = int(f.read().strip())
    try:
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print(f"stopped live server process group of pid {pid}")
    except ProcessLookupError:
        print("process already gone")
